In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/crsp_daily_top500.txt", sep="\t")

/var/folders/r3/t04qzh290tl91rj4_h2w1tlh0000gn/T/ipykernel_34427/1615792969.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/crsp_daily_top500.txt", sep="\t")


## Uncertainty as Historical Volatility

In [5]:
# Sort by firm and date
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

# --- Helper function for rolling skewness ---
def rolling_skew(x):
    x = np.array(x)
    if len(x) < 3 or np.std(x) == 0:
        return np.nan
    m, s = np.mean(x), np.std(x)
    return np.mean(((x - m) / s) ** 3)

# --- Group by permno and create features ---
def make_features(group):
    group = group.sort_values('date').copy()
    
    # Lags
    for i in range(1, 6):
        group[f'ret_lag{i}'] = group['ret'].shift(i)
    group['ret_excess_lag1'] = group['ret_excess'].shift(1)
    
    # Rolling features (past 20 days)
    group['skew_20d'] = group['ret'].rolling(20).apply(rolling_skew, raw=False)
    group['mean_20d'] = group['ret'].rolling(20).mean()
    group['sd_20d'] = group['ret'].rolling(20).std()
    
    # Target: future 20-day volatility (forward-looking window)
    group['vol_20d_forward'] = (
        group['ret']
        .shift(-19)  # aligns current t with volatility t+1:t+20
        .rolling(20)
        .std()
    )
    
    return group

df_features = df.groupby('permno', group_keys=False).apply(make_features).dropna()


/var/folders/r3/t04qzh290tl91rj4_h2w1tlh0000gn/T/ipykernel_34427/1257946186.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_features = df.groupby('permno', group_keys=False).apply(make_features).dropna()


In [ ]:
features = df_features[['ret_lag1', 'ret_excess_lag1', 'ret_lag2', 'ret_lag3', 'ret_lag4', 'ret_lag5',
                        'skew_20d', 'mean_20d', 'sd_20d']]
target = df_features['vol_20d_forward']
df_features.to_csv("../data/crsp_features_full0.txt", sep="\t", index=False)\

## Uncertainty as Implied Volatility (CAPM)

## Uncertainty as Entropy

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import entropy

# --- Load and sort ---
df = pd.read_csv("../data/crsp_daily_top500.txt", sep="\t")
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

# --- Rolling skew helper (same as before) ---
def rolling_skew(x):
    x = np.array(x)
    if len(x) < 3 or np.std(x) == 0:
        return np.nan
    m, s = np.mean(x), np.std(x)
    return np.mean(((x - m) / s) ** 3)

# --- Rolling Shannon entropy helper ---
def rolling_entropy(x, bins=20):
    """Compute Shannon entropy (bits) of returns in a window."""
    hist, _ = np.histogram(x, bins=bins, density=True)
    hist = hist[hist > 0]  # remove zeros
    return entropy(hist, base=2)

# --- Group by firm and create features ---
def make_features(group):
    group = group.sort_values('date').copy()
    
    # Past return lags
    for i in range(1, 6):
        group[f'ret_lag{i}'] = group['ret'].shift(i)
    group['ret_excess_lag1'] = group['ret_excess'].shift(1)
    
    # Past 20-day rolling stats
    group['skew_20d'] = group['ret'].rolling(20).apply(rolling_skew, raw=False)
    group['mean_20d'] = group['ret'].rolling(20).mean()
    group['sd_20d']   = group['ret'].rolling(20).std()
    
    # --- Target: future 20-day entropy ---
    group['entropy_20d_forward'] = (
        group['ret']
        .shift(-19)                # align t with t+1:t+20
        .rolling(20)
        .apply(rolling_entropy, raw=False)
    )
    return group

# --- Apply to all firms ---
df_features_entropy = (
    df.groupby('permno', group_keys=False)
      .apply(make_features)
      .dropna()
)

# --- Save ---
df_features_entropy.to_csv("../data/crsp_features_entropy.txt", sep="\t", index=False)
print("✅ Saved ../data/crsp_features_entropy.txt")


/var/folders/r3/t04qzh290tl91rj4_h2w1tlh0000gn/T/ipykernel_91540/1331864216.py:6: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/crsp_daily_top500.txt", sep="\t")
